<!-- # 网络笔记
## 概述
由于大规模模型端到端训练，视觉和语言预训练的成本越来越高，BLIP-2是一种通用且高效的预训练策略，可以从现成的冻结的预训练图像编码器和冻结的大型语言模型引导视觉语言训练。
### 模型主体框架
<div style="background-color:#f9f9f9; padding:10px; border-radius:5px;">
<image src="./assets/fig1-example.png" />
<span style="font-size:12px; color:#555;">Figure 1. Overview of BLIP-2's framework. We pre-train a lightweight Querying Transformer following a two-stage strategy to bridge the modality gap. The first stage bootstraps vision-language representation learning from a frozen image encoder. The second stage bootstraps vision-to-language generative learning from a frozen LLM, which enables zero-shot instructed image-to-text generation (see Figure 4 for more examples).</span>
</div> 

BLIP-2使用一个轻量级的查询转换器Q-Former来弥补模态差距。该转换器分两个阶段的预训练：第一阶段从冻结的图像编码器引导视觉-语言表征学习；第二阶段从冻结的大型语言模型引导视觉到语言的生成式学习，从而实现了零样本指令式图像到文本生成（更多示例见图4）。

#### 基于冻结图像编码器的视觉语言表征学习阶段
<div style="background-color:#f9f9f9; padding:10px; border-radius:5px;">
    <image src="./assets/fig2-v5.png" />
    <span style="font-size:12px; color:#555;">Figure 2. (<b>Left</b>) Model architecture of Q-Former and BLIP-2's first-stage vision-language representation learning objectives. We jointly optimize three objectives which enforce the queries (a set of learnable embeddings) to extract visual representation most relevant to the text. (<b>Right</b>) The self-attention masking strategy for each objective to control query-text interaction.</span>
</div>

BLIP-2提出Q-Former作为可训练模块，以弥补冻结图像编码器和冻结LLM之间的模态差距。它从图像编码器中提取固定数量的输出特征。Q-Former由两个共享相同自注意力层的transformer子模块组成。一个图像transformer架构和冻结的图像编码器进行交互提取视觉特征；一个文本transformer，既作为文本编码器又作为文本解码器。

- BLIP-2创建了一组可学习查询嵌入(Queries)作为图像transformer的输入。查询通过自注意力层相互交互，并通过交叉注意力层（每隔一个transformer块嵌入）与冻结的图像特征(Image Encoder的输出)进行交互。
- 查询还可以通过相同的自注意力层与文本交互。根据预训练任务的不同，BLIP-2应用不同的注意力掩码来控制查询-文本交互。

#### 损失函数

#### 不同的掩码机制
__图像文本对比学习__ (ITC)

图像文本对比学习学习对齐图像表示和文本表示，使它们的相互信息最大化。它通过对比正对和负对的图像-文本相似性来实现这一点。我们将来自图像transformer 的输出查询表示$Z$与来自文本transformer的文本表示$t$对齐。其中$t$是［cls］令牌的输出嵌入。由于Z包含了多个输出嵌入（每个查询具有一个嵌入），我们首先计算每个查询输出与$t$之间的成对相似度，然后选择最高的一个作为图像-文本相似度。为了避免信息泄露，BLIP-2使用了单模态自注意力掩码，其中查询和文本不允许相互看到。

__基于图像的文本生成__ (ITG)

基于图像的文本生成（ITG）损失训练Q-Former在给定输入图像的条件下生成文本。由于Q-Former的体系结构不允许冻结图像编码器和文本标记之间的直接交互，因此生成文本所需的信息必须首先由查询提取，然后通过自注意力层传递给文本标记。因此，查询被迫提取视觉特征来捕获关于文本的所有信息。BLIP-2使用了一个多模态因果自注意掩码来控制查询-文本交互，查询可以相互关注，但不能关注文本标记。每个文本标记都可以关注所有查询以及其之前的文本标记。BLIP-2还将［cls］令牌替换为一个新的［dec］令牌作为第一个文本令牌来知识解码任务。

__图像文本匹配__ (ITM)

图像文本匹配旨在学习图像和文本表示之间的细粒度对齐，这是一个二元分类任务，要求模型预测图像-文本对是匹配的还是不匹配的。BLIP-2使用双向自注意力掩码，其中所有查询和文本都可以相互关注。输出查询嵌入$Z$从而捕获多模态信息。BLIP-2将每个输出查询嵌入输入到一个两类线性分类器中，得到一个logits，并将所有查询中的logit平均为输出匹配分数。

#### 从冻结的LLM中引导视觉到语言的生成学习

<div style="background-color:#f9f9f9; padding:10px; border-radius:5px;">
    <image src="./assets/fig3-v3.png" />
    <span style="font-size:12px; color:#555;">Figure 3. BLIP-2's second-stage vision-to-language generative pre-training, which bootstraps from frozen large language models (LLMs).
(Top) Bootstrapping a decoder-based LLM (eg. OPT). (Bottom) Bootstrapping an encoder-decoder-based LLM (e.g. FlanT5). The fully-connected layer adapts from the output dimension of the Q-Former to the input dimension of the chosen LLM.</span>
</div>


在生成预训练阶段，BLIP-2将Q-Former（附带冻结图像编码器）连接到冻结的LLM，以获取LLM的生成语言能力。BLIP-2使用全连接（FC）层将输出查询嵌入Z线性投影到与LLM的文本嵌入相同的维度。
然后将投影的查询嵌入预处理为输入文本嵌入。它们起到了软视觉提示的作用，将LLM限制在Q-Former提取的视觉表示上。由于Q-Former经过预训练可以提取语言信息性的视觉表示，它有效的充当了一个信息瓶颈，将最有用的信息提供给LLM，同时去除不相关的视觉信息。这减轻了LLM学习视觉语言对齐的负担，从而减轻了灾难性的遗忘问题。
BLIP-2实验了两种类型的LLMs：基于解码器的LLMs和基于编码器-解码器的LLMS。
- 对于基于解码器的LLM，BLIP-2使用语言建模损失进行预训练，其中冻结的LLM的任务是根据Q-Former的视觉表示生成文本。
- 对于基于编码器-解码器的LLMS，BLIP-2使用前缀语言建模损失进行预训练，BLIP-2将文本分成两部。前缀文本与可视化表示相连接，作为LLM编码器的输入；后缀文本用作LLM解码器的生成目标。 -->

# 论文精读

### 1. 引言 Introduction

- Vision-Language pre-training (VLP) 视觉语言预训练模型发展很快，但大部分是端到端训练，计算资源消耗大。
- 我们推出了一个通用且高效的预训练策略BLIP-2，并冻结预训练的模型来节约计算成本并预防灾难性遗忘。
- LLMs看不见图像，冻结他们使对齐更加困难，现有的方法（Frozen, Flamingo）使用image-to-text生成损失，这被证明是不够的。
- 推出Querying Transformer(Q-Former)来对齐冻结的俩个模型

- 我们的VLP(BLIP-2)的优势:
    - 有效利用了俩模型，在多个vision-language任务上取得先进成绩。
    - 由大语言模型驱动，实现了强大的零样本图像到文本生成能力。
    - 冻结模型且使用轻量的Q-Former，计算高效。（高8.7% Flemingo, 少54倍的学习参数）

### 2. 相关工作 Related Work

__端到端的视觉-语言预训练__

- 大多数VLP使用大规模的图像-文本对进行端到端预训练
  1. 计算成本高。
  2. 且灵活性不足（仅利用了单个预训练模型）

### 3. 方法论 Method

<div style="background-color:#f9f9f9; padding:10px; border-radius:5px;">
<image src="./assets/fig1-example.png" />
<span style="font-size:12px; color:#555;">Figure 1. Overview of BLIP-2's framework. We pre-train a lightweight Querying Transformer following a two-stage strategy to bridge the modality gap. The first stage bootstraps vision-language representation learning from a frozen image encoder. The second stage bootstraps vision-to-language generative learning from a frozen LLM, which enables zero-shot instructed image-to-text generation (see Figure 4 for more examples).</span>
</div> 

BLIP-2使用一个轻量级的查询转换器Q-Former来弥补模态差距。该转换器分两个阶段的预训练：第一阶段从冻结的图像编码器引导视觉-语言表征学习；第二阶段从冻结的大型语言模型引导视觉到语言的生成式学习，从而实现了零样本指令式图像到文本生成（更多示例见图4）。

- 两阶段预训练的Q-Fromer:
    1. vision-language representation learning 视觉语言表征学习(用冻结的图像编码器)
       - 用三个loss训练Q-Former，让queries提取与文本最相关的信息
    2. vision-to-language generative learning 视觉到语言生成学习(用冻结的LLM)
       - queries通过FC层投影到LLM的文本嵌入维度，作为软视觉提示，指导LLM生成文本

#### 3.1 模型架构 Model Architecture

#### 3.2 基于冻结图像编码器的视觉语言表征学习阶段 Bootstrap Vision-Language Representation Learning from a Frozen Image Encoder

<div style="background-color:#f9f9f9; padding:10px; border-radius:5px;">
    <image src="./assets/fig2-v5.png" />
    <span style="font-size:12px; color:#555;">Figure 2. (<b>Left</b>) Model architecture of Q-Former and BLIP-2's first-stage vision-language representation learning objectives. We jointly optimize three objectives which enforce the queries (a set of learnable embeddings) to extract visual representation most relevant to the text. (<b>Right</b>) The self-attention masking strategy for each objective to control query-text interaction.</span>
</div>

- 两个主要组件:
    1. image transformer 图像转换器: 用于从冻结的图像编码器提取视觉表示
    2. text transformer 文本转换器: 既作为文本编码器又作为文本解码器
- 两个组件的Self-Attention层是共享的($W_q, W_k, W_v$)
- 使用BERT-base初始化Q-Former，cross-atten则随机初始化
- 实验中，使用了32个查询嵌入(每个768维)，输出查询标志$Z$(32 x 768)远小于冻结的图像特征(257 x 1024)

```
[query, text] => Masked Self-Attention => [query] => Cross-Attention => FeedForward1 => [query, text] 
                                       => [text]                     => FeedForward2 
```

<font color="green">

与BLIP刚好相反，BLIP中共享的是Feed-Forward层，Self-Attention层是独立的。

</font>

<!-- GPT: 三个不同的目标是并行计算的，最后在反向传播的时候使用了加权和计算总损失。 -->


__图像文本对比学习 (Image-Text Contrastive Learning ITC)__

图像文本对比学习学习对齐图像表示和文本表示，使它们的互信息最大化。它通过对比正对和负对的图像-文本相似性来实现这一点。我们将来自图像transformer的输出查询表示$Z$与来自文本transformer的文本表示$t$对齐。其中$t$是［cls］令牌的输出嵌入。由于$Z$包含了多个输出嵌入（每个查询具有一个嵌入），我们首先计算每个查询输出与$t$之间的成对相似度，然后选择最高的一个作为图像-文本相似度。为了避免信息泄露，BLIP-2使用了单模态自注意力掩码，其中查询和文本不允许相互看到。

__基于图像的文本生成 (Image-grounded Text Generation ITG)__

基于图像的文本生成（ITG）损失训练Q-Former在给定输入图像的条件下生成文本。由于Q-Former的体系结构不允许冻结图像编码器和文本标记之间的直接交互，因此生成文本所需的信息必须首先由查询提取，然后通过自注意力层传递给文本标记。因此，查询被迫提取视觉特征来捕获关于文本的所有信息。BLIP-2使用了一个多模态因果自注意掩码来控制查询-文本交互，查询可以相互关注，但不能关注未来的文本标记。每个文本标记都可以关注所有查询以及其之前的文本标记。BLIP-2还将［cls］令牌替换为一个新的［dec］令牌作为第一个文本令牌来知识解码任务。

[32 queries, dec, t1, t2, ..., ti-1] 预测 ti

__图像文本匹配 (Image-Text Matching ITM)__

图像文本匹配旨在学习图像和文本表示之间的细粒度对齐，这是一个二元分类任务，要求模型预测图像-文本对是匹配的还是不匹配的。BLIP-2使用双向自注意力掩码，其中所有查询和文本都可以相互关注。输出查询嵌入$Z$从而捕获多模态信息。BLIP-2将每个输出查询嵌入输入到一个两类线性分类器中，得到一个logits，并将所有查询中的logit平均为输出匹配分数。

Cross Entropy on mean(FC(queries)) -- FC: 2类线性分类器

```py
v1_embeddings = output_itm.last_hidden_state[:, : query_tokens_itm.size(1), :]
v1_output = self.itm_head(v1_embeddings) 
logits = v1_output.mean(dim=1)
```

#### 3.3 从冻结的LLM中引导视觉到语言的生成学习 Bootstrap Vision-to-Language Generative Learning from a Frozen LLM

<div style="background-color:#f9f9f9; padding:10px; border-radius:5px;">
    <image src="./assets/fig3-v3.png" />
    <span style="font-size:12px; color:#555;">Figure 3. BLIP-2's second-stage vision-to-language generative pre-training, which bootstraps from frozen large language models (LLMs).
(Top) Bootstrapping a decoder-based LLM (eg. OPT). (Bottom) Bootstrapping an encoder-decoder-based LLM (e.g. FlanT5). The fully-connected layer adapts from the output dimension of the Q-Former to the input dimension of the chosen LLM.</span>
</div>

使用一个全连接(FC)层将输出的查询嵌入$Z$(32 x 768)线性投影到与LLM的文本嵌入相同的维度(eg. 2048)。

- 对于仅解码器LLM: 
  - [queries_projectd] 作为输入，预测文本序列 [t1, t2, ..., tk]
  - 损失: 整个文本序列
- 对于编解码器LLM: 
  - [queries_projectd, t1, t2, ..., ti-1] 作为编码器输入
  - [ti, ..., tk] 作为解码器目标
  - 损失: 只需要计算解码器输出

- 还有两张描述两个阶段的图:


<div style="display: flex; justify-content: center;">
    <image src="./assets/stage1.png" style="width: 80%; border-radius:5px;" />
</div>


<div style="display: flex; justify-content: center;">
    <image src="./assets/stage2.png" style="width: 80%; border-radius:5px;" />
</div>

#### 3.4 模型预训练 Model Pre-training

__预训练数据 Pre-training Data__

- 使用和BLIP相同的129M图像。
    - COCO
    - Visual Genome
    - CC3M
    - CC12M
    - SBU
    - 115M from LAION-400M
- 用`CapFilt`方法来创建人工注解(synthetic captions),
  - 使用BLIP_large模型生成captions，通过CLIP ViT-L/14模型计算相似度。每次在最高的2个中随机选择一个。


__Pre-trained image encoder and LLM__

- 尝试了两个SOTA的预训练vision transformer:
  - ViT-L/14 from CLIP
  - ViT-g/14 from EVA-CLIP
- 移除了ViT的最后一层，用倒数第二层表示特征
- 尝试了两种LLM:
  - Decoder-based: OPT
  - Encoder-decoder-based: FlanT5

__预训练设置 Pre-training Settings__

- 一阶段`250K`步，二阶段`80K`步
- batch size:
  - 2320/1680 for ViT-L/ViT-g
  - 1920/1520 for OPT/FlanT5
- FP16(Except for FlanT5 uses BFloat16)

- 16-A100(40G):最大的模型使用Vit-g和FlanT5-XXL, 6(stage 1)+3(stage 2)
- 优化器：AdamW($\beta_1=0.9, \beta_2=0.98$, decay=0.05)
- 学习率：cosine decay with 2000 warm-up steps
  - peak: 1e-4
  - minimum lr at stage 2: 5e-5
- 图像处理: 尺寸 224 x 224， 随机裁剪，水平翻转

### 4. 实验 Experiments

### 5. 局限 Limitations

- 最新的LLM可以在少样本情况下进行上下文学习，但BLIP-2在VQA任务中缺乏这个能力。
- 归因于预训练的数据：没有提供这种上下文学习的能力。（每个只有图像-文本对）
- 未来可能会推出这样的数据集

- 图像-文本生成可能不理想：
  - LLM的知识不准确
  - 触发错误的推理路径
  - 缺乏图像中的最新内容
- 由于使用冻结的LLM，还可能存在冒犯性语言、社会偏见、隐私泄漏等问题。
- 未来可能使用过滤的数据集。

#### 补充对比（来自bilibili）

- 都是用了Frozen的Image Encoder（Clip）（Flamingo是NFNet）
- 都接LLM
- BLIP-2：Q-Former， CA， queries， FC， prefix
  - 3 losses pretraining
  - LLaVA Step 1
- LLaVA： FC/MLP：
  - LLM but only train a adapters
  - LLM FT
